# Demo 2 — pgvector: the database scores and sorts

In demo 1, Python computed the similarity scores and sorted the results.
Here that job moves to PostgreSQL: Python only generates the embeddings, and the
**database** computes the distances and returns the rows already ordered — plain SQL.

Start the database first:

```bash
docker compose up -d
```

In [ ]:
import psycopg
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
conn = psycopg.connect(
    "host=localhost port=5433 dbname=rag user=postgres password=postgres",
    autocommit=True,
)

## Step 1 — Embed the sentences and store them in PostgreSQL

Each row gets the original text plus its embedding in a `VECTOR(384)` column
(see `init.sql`). pgvector accepts the vector as a string like `[0.1, 0.2, ...]`.

In [ ]:
sentences = [
    "The dog barks loudly",
    "A cat sleeps on the sofa",
    "Puppies love to play",
    "The stock market crashed",
    "Interest rates went up",
]

embeddings = model.encode(sentences)

with conn.cursor() as cur:
    cur.execute("TRUNCATE documents")
    for text, embedding in zip(sentences, embeddings):
        cur.execute(
            "INSERT INTO documents (content, embedding) VALUES (%s, %s)",
            (text, str(embedding.tolist())),
        )

with conn.cursor() as cur:
    cur.execute("SELECT id, content, embedding::text FROM documents")
    for id_, content, embedding in cur.fetchall():
        print(f"{id_}  {content}  {embedding[:40]}...")

## Step 2 — Embed the query

The query becomes one more vector, generated with the same model.

In [ ]:
query = "My puppy is making noise"
query_embedding = str(model.encode(query).tolist())

## Step 3 — Let SQL do the scoring and the sorting

No Python math here: `<=>` is pgvector's **cosine distance** operator, and the
`ORDER BY ... LIMIT` runs inside PostgreSQL. The rows come back already ranked.

In [ ]:
sql = """
SELECT content,
       1 - (embedding <=> %(q)s::vector) AS cosine_similarity
FROM documents
ORDER BY embedding <=> %(q)s::vector
LIMIT 3
"""

with conn.cursor() as cur:
    cur.execute(sql, {"q": query_embedding})
    for content, similarity in cur.fetchall():
        print(f"{similarity:.3f}\t{content}")

## Step 4 — Same thing with Euclidean distance

pgvector also has `<->`, the **Euclidean distance** from Part 2 of the README.
Lower distance = better match, so the ordering is the same idea, flipped.

In [ ]:
sql = """
SELECT content,
       embedding <-> %(q)s::vector AS euclidean_distance
FROM documents
ORDER BY euclidean_distance
LIMIT 3
"""

with conn.cursor() as cur:
    cur.execute(sql, {"q": query_embedding})
    for content, distance in cur.fetchall():
        print(f"{distance:.3f}\t{content}")